# 04 — Backtest Analysis

> **Educational research only.** Synthetic data, simulated results, not financial advice.

OOS predictions → signals → portfolio → self-financing future-open execution → deflated Sharpe.

In [ ]:
import matplotlib.pyplot as plt

from alphaforge.backtesting import run_backtest
from alphaforge.data import SyntheticMarketConfig, generate_synthetic_market
from alphaforge.features import build_features
from alphaforge.labels import build_labels
from alphaforge.portfolio import construct_portfolio
from alphaforge.risk import performance_summary
from alphaforge.signals import build_signals, select_model_predictions
from alphaforge.training import run_walk_forward

panel = generate_synthetic_market(SyntheticMarketConfig(n_symbols=10, n_days=1500, seed=42))
features = build_features(panel, benchmark_symbol='BENCH')
labels = build_labels(panel, benchmark_symbol='BENCH', horizons=[1, 5, 20])
result = run_walk_forward(features, labels, [{'name': 'ridge', 'params': {'alpha': 10.0}}],
                          target='fwd_ret_5',
                          config={'min_train_days': 504, 'test_days': 126, 'step_days': 126, 'embargo_days': 21},
                          max_horizon=20)
selected = select_model_predictions(result.predictions)
signals = build_signals(selected, strategy='long_short', params={'quantile': 0.2})
weights = construct_portfolio(signals, features=features, config={'max_weight': 0.15})
bt = run_backtest(
    panel,
    weights,
    benchmark_symbol='BENCH',
    rebalance_frequency=5,
    costs={'commission_bps': 1.0, 'half_spread_bps': 2.5, 'slippage_bps': 2.0},
    risk={'vol_target': 0.10},
    liquidate_at_end=True,
)
performance_summary(bt.equity_curve)

In [ ]:
active = bt.equity_curve[bt.equity_curve['active']]
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(active['date'], active['equity'])
axes[0].set_title('Equity (net of costs, simulated)')
axes[1].plot(active['date'], active['gross_exposure'])
axes[1].set_title('Gross exposure (vol-targeted)')
plt.tight_layout()

In [ ]:
from alphaforge.evaluation import deflated_sharpe_ratio

deflated_sharpe_ratio(active['return'], n_trials=4)

`n_trials` must count every variant that competed for selection — understating it is how backtests lie. See docs/modeling.md.